# 1. Imports

In [1]:
import pandas as pd
import numpy as np
import polars as pl
import pyarrow as pa

import glob
import json
from pathlib import Path

import calendar
import geopandas as gpd
from geodatasets import get_path
from shapely.geometry import Point
import branca.colormap as cm

import folium
from folium.plugins import MarkerCluster
import folium
from folium.plugins import MarkerCluster
from folium.elements import MacroElement
from jinja2 import Template

DATA_PATH = Path("../data/UpdatedData/citibike_2023_combined.parquet/")
OUTPUT_PATH = Path("../output/")

# 2. Data Load & Process

Processing script should be in `preprocessing.ipynb`

In [2]:
df = (
    pl.read_parquet(f"{DATA_PATH}/*.parquet")
    # 1) Parse to datetime and rename in one go
    .with_columns(
        pl.col("started_at").str.to_datetime().alias("start_time"),
        pl.col("ended_at").str.to_datetime().alias("end_time"),
    )
    # 2) Compute ride_time_seconds = end_time - start_time (in seconds)
    .with_columns(
        (pl.col("end_time") - pl.col("start_time"))
        .dt.total_seconds()
        .alias("ride_time_seconds")
    )
    # 3) Order by start_time
    .sort("start_time")
    # 4) Select columns in the exact order you want
    .select(
        "ride_id",
        "rideable_type",
        "start_time",
        "end_time",
        "ride_time_seconds",   # ← right after end_time
        "start_station_name",
        "start_station_id",
        "end_station_name",
        "end_station_id",
        "start_lat",
        "start_lng",
        "end_lat",
        "end_lng",
        "member_casual",
    )
)

# remove any start or end_station_name = null
df = df.filter(pl.col("end_station_name").is_not_null())
df= df.filter(pl.col("end_station_id").is_not_null())
df = df.filter(pl.col("start_station_name").is_not_null())
df= df.filter(pl.col("start_station_id").is_not_null())

# filter data out for ride_time < 2 hours or way too short ones like 1 minute
df = df.filter(pl.col("ride_time_seconds") < 2 * 3600)   
df = df.filter(pl.col("ride_time_seconds") >= 60)

df = df.filter(pl.col("start_time").dt.year() == 2023)
df = df.with_columns([
    pl.col("start_time").dt.month().alias("month"),
])

# filter data out for ride time that is negative or zero (not possible)
df = df.filter(pl.col("ride_time_seconds") > 0)

# Remove rides with identical start/end stations (probably some error)
df = df.filter(
    ~(
        (pl.col("start_station_id") == pl.col("end_station_id"))
    )
)

df = df.filter(
    ~(
        (pl.col("start_lat") == pl.col("end_lat"))
        & (pl.col("start_lng") == pl.col("end_lng"))
    )
)

In [3]:
df = df.to_pandas()        # main pandas DataFrame

In [32]:
df.head()

,ride_id,rideable_type,start_time,end_time,ride_time_seconds,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,month
0,4612E3DBB1F3088D,electric_bike,2023-01-01 00:00:13.021,2023-01-01 00:19:55.832,1182,W 56 St & 10 Ave,6955.01,Harrison St & Hudson St,5400.05,40.768254,-73.988639,40.718710,-74.009001,member,1
1,2054AE257784A52C,electric_bike,2023-01-01 00:00:27.436,2023-01-01 00:06:50.763,383,President St & Henry St,4307.13,Clinton St & 4 Place,4119.04,40.682800,-73.999904,40.678356,-74.000145,member,1
2,EB0C899F37A980A4,electric_bike,2023-01-01 00:00:33.209,2023-01-01 01:22:53.015,4939,Centre St & Chambers St,5207.01,W 48 St & Rockefeller Plaza,6626.11,40.712733,-74.004607,40.757769,-73.979294,casual,1
3,FFA9DAE23C0676C7,electric_bike,2023-01-01 00:00:49.245,2023-01-01 00:03:49.355,180,West End Ave & W 78 St,7340.07,Central Park West & W 76 St,7253.04,40.783786,-73.981687,40.778968,-73.973747,member,1
4,8FAADDF099429A49,electric_bike,2023-01-01 00:01:50.411,2023-01-01 00:05:23.478,213,Amsterdam Ave & W 66 St,7149.05,W 64 St & Thelonious Monk Circle,7123.04,40.774667,-73.984706,40.775160,-73.989187,member,1


# 3. Visualization

### 3.1. NYC Bike Share Choropleth Map: Geographic Distribution of Bike Rides

I don't know what our questions will be so i just wrote random things to help you guys determine what we want to do with your vis,

We wanted to understand where most bike share rides start in NYC. This helps us see which neighborhoods use the system most and could guide decisions about where to add more bike stations or improve service.

**Question: Which ZIP codes generate the most bike share rides?**
We created a choropleth map showing the number of rides starting in each ZIP code. We chose ZIP codes because they represent recognizable neighborhoods and are large enough to show clear patterns without too much detail. Light blue to dark blue gradient for areas with rides (darker = more rides). Light grey for ZIP codes with zero rides

The choropleth reveals clear geographic patterns in bike share usage across New York City. Manhattan dominates the ridership landscape, with the darkest blue areas concentrated in Midtown, Lower Manhattan, and residential neighborhoods like the Upper West Side. This pattern suggests that Manhattan serves as the primary hub for bike share activity, likely driven by its dense mixture of office buildings, tourist destinations, and residential areas.

Brooklyn shows strong usage patterns as well, particularly in neighborhoods like Williamsburg, Downtown Brooklyn, and Park Slope. These areas display notably high ridership, indicating that Brooklyn has successfully integrated bike sharing into its transportation ecosystem. Queens demonstrates moderate usage with scattered pockets of ridership throughout accessible neighborhoods, though the intensity remains less pronounced than in Manhattan or Brooklyn. The Bronx shows limited coverage with few active ZIP codes, suggesting that the bike share system has less presence in this borough. Finally, Staten Island appears entirely grey on the map, indicating that Citi Bike does not currently operate in this borough.

This visualization has several limitations that suggest directions for deeper analysis. First, the map shows only where rides start, not where they end. Understanding destination patterns would reveal important travel corridors, commuter routes, and how riders actually move through the city. Second, the map averages ride data across all times and seasons, which masks potentially important temporal variations. We cannot tell from this visualization whether patterns differ between weekdays and weekends, or how ridership changes from summer to winter months.

Finally, ZIP codes vary considerably in geographic size, which means larger ZIP codes might appear to have more rides simply because they cover more area, not because of higher ridership intensity. Despite these limitations, this map establishes an important foundation by showing where the bike share system is most actively used, setting the stage for future visualizations that could explore when people ride, where they go, and who is riding.

In [4]:
df_choropleth = df[['rideable_type', 'ride_time_seconds', 'month','member_casual', 'start_lat', 'start_lng']].copy()

In [5]:
# Load zip code boundaries
zips = gpd.read_file("../data/nyc_zipcode.geojson")  # path to your ZIP shapefile
zips = zips.to_crs(epsg=4326)

# Convert df_points to GeoDataFrame
geometry = [Point(xy) for xy in zip(df_choropleth['start_lng'], df_choropleth['start_lat'])]
gdf = gpd.GeoDataFrame(df_choropleth, geometry=geometry, crs="EPSG:4326")

# Join ride data with zip code boundaries
joined_gdf = gpd.sjoin(gdf, zips[['postalCode', 'geometry', 'borough']], how="left", predicate="within")

joined_gdf.head()

,rideable_type,ride_time_seconds,month,member_casual,start_lat,start_lng,geometry,index_right,postalCode,borough
0,electric_bike,1182,1,member,40.768254,-73.988639,POINT (-73.98864 40.76825),203.0,10019,Manhattan
1,electric_bike,383,1,member,40.682800,-73.999904,POINT (-73.9999 40.6828),150.0,11231,Brooklyn
2,electric_bike,4939,1,casual,40.712733,-74.004607,POINT (-74.00461 40.71273),129.0,10007,Manhattan
3,electric_bike,180,1,member,40.783786,-73.981687,POINT (-73.98169 40.78379),74.0,10024,Manhattan
4,electric_bike,213,1,member,40.774667,-73.984706,POINT (-73.98471 40.77467),89.0,10023,Manhattan


In [6]:
#  Base rides layer
zip_counts_all = (
    joined_gdf.groupby("postalCode")
              .size()
              .reset_index(name="num_rides")
)

# Merge counts back onto the ZIP SHAPES (this step keeps the polygons)
choropleth_all = zips.merge(zip_counts_all, on="postalCode", how="left")
choropleth_all["num_rides"] = choropleth_all["num_rides"].fillna(0)

In [ ]:
# Create a color map for >0 values
max_val = choropleth_all['num_rides'].max()

colormap = cm.LinearColormap(
    colors=["#deebf7", "#3182bd"],   # light blue → dark blue
    vmin=1, vmax=max_val
)

# Function to assign color
def style_function(feature):
    val = feature['properties']['num_rides']
    if val == 0:
        return {
            'fillColor': '#d3d3d3',   # light grey for zero
            'color': 'black',
            'fillOpacity': 0.5,
            'weight': 0.7
        }
    else:
        return {
            'fillColor': colormap(val),
            'color': 'black',
            'fillOpacity': 0.8,
            'weight': 0.7
        }

# -----------------------------------
# Make map + tooltip
# -----------------------------------
m = folium.Map(location=[40.75, -73.97], zoom_start=11, tiles="cartodbpositron")

tooltip = folium.GeoJsonTooltip(
    fields=["postalCode", "borough", "num_rides"],
    aliases=["ZIP Code:", "Borough:", "Number of rides:"],
    localize=True,
    sticky=False,
    labels=True,
    style=(
        "background-color: white; "
        "color: #333333; "
        "font-size: 12px; "
        "padding: 4px; "
        "border-radius: 3px;"
    )
)

# Use only needed columns in geo_data for clarity
geo_all = choropleth_all[["postalCode", "borough", "num_rides", "geometry"]]

geojson_all = folium.GeoJson(
    geo_all,
    name="All rides",
    style_function=style_function,
    tooltip=tooltip
).add_to(m)

# Add the color scale legend
colormap.caption = "Number of Rides Starting in ZIP Code"
colormap.add_to(m)

# Layer control + save
folium.LayerControl().add_to(m)

m.save("../output/2023_NYC_bike_choropleth.html")

### 3.2. NYC Interactive Bike Share Map

For "motivation" section of the final project, this could be something like... (Choose a question to answer here:)

After examining the choropleth map showing the spatial distribution of ride starting locations and analyzing temporal patterns through our previous visualizations, we identified a critical limitation: while we could see where rides begin and when they occur, we had no clear view of where riders actually go. The choropleth map effectively shows that the majority of rides originate in Manhattan and parts of Brooklyn, and our XXXXX (OTHER VISUALIZATION) analyses reveal XXXXX. However, these static visualizations left a fundamental question unanswered: What are the destination patterns for rides originating from different stations?

Understanding origin-destination relationships is crucial for several reasons. For urban planners, it reveals commuter corridors and recreational routes that may need infrastructure improvements. For Citi Bike operations, it helps predict bike redistribution needs. It allows them to make key business decisions by understanding which stations consistently send bikes to specific destinations allows for proactive rebalancing. For researchers studying urban mobility, these patterns illuminate how residents and visitors actually move through the city.

This motivated us to develop a fully interactive map-based visualization that allows users to explore ride destinations from any starting station in the NYC Citi Bike network. While static visualizations effectively show aggregate patterns, they cannot capture the dynamic, exploratory nature of origin-destination relationships that are central to understanding bike share usage patterns.

The interactive map addresses several key questions that static visualizations cannot:

1. Destination diversity: Where do riders typically go from a particular station? Are destinations concentrated in specific areas (suggesting commuter hubs or specific-purpose origins), or do they spread across the city (indicating more general-purpose stations)?
2. Trip characteristics: What are the typical ride durations between different origin-destination pairs? How do these patterns vary by bike type (classic vs. electric) or rider type (member vs. casual)? For instance, do electric bikes enable longer-distance trips?
3. Temporal variations: How do destination patterns and ride characteristics change across different months of the year?

The NYC Citi Bike dataset contains over 1,000,000 rides per month, which presents both computational and visualization challenges. Rendering hundreds of thousands of individual ride endpoints would create visual clutter, slow performance, and ironically make patterns harder to see rather than easier. To address this, we implemented a stratified random sampling approach, selecting 100,000 rides per month.

In [9]:
# for each months, get 100,000 rides randomly
df_reduced = df.groupby('month').apply(lambda x: x.sample(n=100000, random_state=42)).reset_index(drop=True)

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_26216\4010079171.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_reduced = df.groupby('month').apply(lambda x: x.sample(n=100000, random_state=42)).reset_index(drop=True)


In [10]:
class RideHighlightJS(MacroElement):
    _template = Template(u"""
        {% macro script(this, kwargs) %}

        // --------------------------------------------------
        // Load noUiSlider (for dual-handle duration slider)
        // --------------------------------------------------
        (function() {
            // CSS
            var link = document.createElement('link');
            link.rel = 'stylesheet';
            link.href = "https://cdnjs.cloudflare.com/ajax/libs/noUiSlider/15.7.2/nouislider.min.css";
            document.head.appendChild(link);

            // JS
            var script = document.createElement('script');
            script.src = "https://cdnjs.cloudflare.com/ajax/libs/noUiSlider/15.7.2/nouislider.min.js";
            document.head.appendChild(script);

            // store to use later in onload
            window._noUiSliderScriptRef = script;
        })();


        // --------------------------------------------------
        // DATA FROM PYTHON
        // --------------------------------------------------
        var ridesByStart   = {{ this.rides_by_start   | safe }};
        var selectedPopups = {{ this.selected_popups | safe }};
        var map            = {{ this._parent.get_name() }};
        var clusterLayer   = {{ this.cluster_name }};
        var globalMinSec   = {{ this.min_sec }};
        var globalMaxSec   = {{ this.max_sec }};
        var monthsList     = {{ this.months | safe }};


        // --------------------------------------------------
        // GLOBAL STATE
        // --------------------------------------------------
        window._allStartMarkers     = window._allStartMarkers     || [];
        window._activeEndpoints     = window._activeEndpoints     || [];
        window._selectedStartMarker = window._selectedStartMarker || null;
        window._lastStartId         = window._lastStartId         || null;
        window._lastStartMarker     = window._lastStartMarker     || null;

        // Current filter state
        var filterState = {
            minSec: globalMinSec,
            maxSec: globalMaxSec,
            bikeTypes: {
                classic_bike: true,
                electric_bike: true
            },
            memberTypes: {
                member: true,
                casual: true
            },
            month: "ALL"   // controls which month is active
        };


        // --------------------------------------------------
        // FILTER HELPERS
        // --------------------------------------------------
        function passesFilter(pt) {
            // bike type filter
            var bt = (pt.bike_type || "").toLowerCase();
            if (bt.includes("classic") && !filterState.bikeTypes.classic_bike) return false;
            if (bt.includes("electric") && !filterState.bikeTypes.electric_bike) return false;

            // member type filter
            var mt = (pt.member || "").toLowerCase();
            if (mt === "member" && !filterState.memberTypes.member) return false;
            if (mt === "casual" && !filterState.memberTypes.casual) return false;

            // month filter
            if (filterState.month !== "ALL") {
                if (String(pt.month) !== String(filterState.month)) {
                    return false;
                }
            }

            return true;
        }


        // Does this station (startId) have any rides in the selected month?
        function stationMatchesCurrentFilters(startId) {
            var pts = ridesByStart[startId] || [];
            for (var i = 0; i < pts.length; i++) {
                if (passesFilter(pts[i])) {   // uses bike/member/month logic
                    return true;
                }
            }
            return false;
        }

        // Show only those start markers whose rides pass current filters
        function filterStartsByFilters() {
            // Make sure cluster layer is on the map
            if (!map.hasLayer(clusterLayer)) {
                map.addLayer(clusterLayer);
            }

            // Always start from a clean cluster state
            clusterLayer.clearLayers();

            window._allStartMarkers.forEach(function(m) {
                var sid = m._startId;
                if (!sid) return;

                if (stationMatchesCurrentFilters(sid)) {
                    clusterLayer.addLayer(m);
                    if (m.setOpacity) m.setOpacity(1.0);
                } else {
                    //Optional: faint style if they ever reappear
                    if (m.setOpacity) m.setOpacity(0.3);
                }
            });
        }


        // --------------------------------------------------
        // MAIN HIGHLIGHT LOGIC
        // --------------------------------------------------
        function highlightStart(startId, marker) {
            // When a station is clicked, remove the cluster layer
            if (map.hasLayer(clusterLayer)) {
                map.removeLayer(clusterLayer);
            }

            // Fade all other start markers (not super visible after layer removal,
            // but harmless to keep)
            window._allStartMarkers.forEach(function(m) {
                if (m !== marker && m.setOpacity) {
                    m.setOpacity(0.3);
                }
            });
            if (marker.setOpacity) {
                marker.setOpacity(1.0);
            }

            // Remove previous selected start marker
            if (window._selectedStartMarker) {
                map.removeLayer(window._selectedStartMarker);
                window._selectedStartMarker = null;
            }

            // Add non-clustered selected start marker with custom bike icon
            window._selectedStartMarker = L.marker(marker.getLatLng(), {
                icon: L.icon({
                    iconUrl: "{{ this.bike_icon_url }}",
                    iconSize: [32, 32],
                    iconAnchor: [16, 32]
                }),
                zIndexOffset: 2000
            }).addTo(map);

            // Bind popup (same content as original start marker popup)
            if (selectedPopups[startId]) {
                window._selectedStartMarker.bindPopup(selectedPopups[startId]);
            }

            // Remove any existing endpoint markers
            window._activeEndpoints.forEach(function(layer) {
                map.removeLayer(layer);
            });
            window._activeEndpoints = [];

            // ─────────────────────────────────────────────
            // 1) Aggregate rides by endpoint, applying ONLY non-time filters
            // ─────────────────────────────────────────────
            var pts = ridesByStart[startId] || [];

            var destAgg = {};  // key -> {lat, lng, name, count, totalSec}

            pts.forEach(function(pt) {
                if (!passesFilter(pt)) return;  // bike / member / month filters

                var name = pt.name || "";
                var key = pt.lat.toFixed(5) + "," + pt.lng.toFixed(5) + "|" + name;

                if (!destAgg[key]) {
                    destAgg[key] = {
                        lat: pt.lat,
                        lng: pt.lng,
                        name: name,
                        count: 0,
                        totalSec: 0
                    };
                }

                destAgg[key].count += 1;
                destAgg[key].totalSec += pt.sec;
            });

            // ─────────────────────────────────────────────
            // 2) Create one marker per destination, filtering by *average* duration
            // ─────────────────────────────────────────────
            Object.keys(destAgg).forEach(function(key) {
                var d = destAgg[key];

                if (d.count === 0) return;

                var avgSec = d.totalSec / d.count;

                // Filter by slider range: average duration in seconds
                if (avgSec < filterState.minSec || avgSec > filterState.maxSec) {
                    return;  // skip this destination
                }

                var avgMin = avgSec / 60.0;

                var tooltipText =
                    (d.name ? (d.name + "<br>") : "") +
                    "Rides ending here: " + d.count + "<br>" +
                    "Avg duration: " + avgMin.toFixed(1) + " min<br>" +
                    "(" + d.lat.toFixed(5) + ", " + d.lng.toFixed(5) + ")";

                var endMarker = L.marker([d.lat, d.lng], {
                    icon: L.icon({
                        iconUrl: "{{ this.flag_icon_url }}",
                        iconSize: [26, 26],
                        iconAnchor: [13, 26]
                    })
                }).addTo(map);

                endMarker.bindTooltip(tooltipText, {
                    permanent: false,
                    direction: 'top',
                    offset: [0, -10]
                });

                window._activeEndpoints.push(endMarker);
            });
        }



        // --------------------------------------------------
        // RESET MAP STATE (spatial only)
        // --------------------------------------------------
        function resetMap() {
            // Add cluster layer back if missing
            if (!map.hasLayer(clusterLayer)) {
                map.addLayer(clusterLayer);
            }

            // Remove all endpoint markers
            window._activeEndpoints.forEach(function(layer) {
                map.removeLayer(layer);
            });
            window._activeEndpoints = [];

            // Remove selected start marker
            if (window._selectedStartMarker) {
                map.removeLayer(window._selectedStartMarker);
                window._selectedStartMarker = null;
            }

            window._lastStartId = null;
            window._lastStartMarker = null;
        }


        // --------------------------------------------------
        // FILTER CONTROL (DUAL SLIDER + CHECKBOXES + MONTH + RESET)
        // --------------------------------------------------
        var filterCtrl = L.control({position: 'topright'});
        filterCtrl.onAdd = function(map_) {
            var div = L.DomUtil.create('div', 'filter-control');
            div.style.background    = "white";
            div.style.padding       = "8px 10px";
            div.style.border        = "1px solid #666";
            div.style.borderRadius  = "6px";
            div.style.boxShadow     = "0 2px 6px rgba(0,0,0,0.3)";
            div.style.fontSize      = "12px";
            div.style.maxWidth      = "260px";

            div.innerHTML = `
                <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:4px;">
                    <div style="font-weight:600;">Filters</div>
                </div>

                <!-- Ride duration dual slider -->
                <div style="margin-bottom:8px;">
                    <label>Avg Ride Duration (sec)</label>
                    <div id="timeSlider" style="margin-top:8px;"></div>
                    <div id="timeRangeLabel"
                         style="font-size:11px;margin-top:6px;font-weight:600;">
                        ${globalMinSec} – ${globalMaxSec}
                    </div>
                </div>

                <!-- Bike type -->
                <div style="margin-bottom:6px;">
                    <div style="font-weight:500;">Bike type</div>
                    <label><input type="checkbox" id="chkClassic" checked> Classic</label><br>
                    <label><input type="checkbox" id="chkElectric" checked> Electric</label>
                </div>

                <!-- Rider type -->
                <div style="margin-bottom:6px;">
                    <div style="font-weight:500;">Rider type</div>
                    <label><input type="checkbox" id="chkMember" checked> Member</label><br>
                    <label><input type="checkbox" id="chkCasual" checked> Casual</label>
                </div>

                <!-- Month dropdown -->
                <div style="margin-top:6px;margin-bottom:4px;">
                    <div style="font-weight:500;">Month</div>
                    <select id="monthSelect" style="width:100%;margin-top:4px;">
                        <option value="ALL">All months</option>
                        ${monthsList.map(function(m) {
                            return '<option value="' + m + '">' + m + '</option>';
                        }).join('')}
                    </select>
                </div>
                         
                <button id="resetMapBtn"
                    style="
                        padding: 3px 8px;
                        background: #f5f5f5;
                        border: 1px solid #888;
                        border-radius: 4px;
                        cursor: pointer;
                        font-size: 11px;
                    ">
                    Reset
                </button>
            `;

            L.DomEvent.disableClickPropagation(div);
            return div;
        };
        filterCtrl.addTo(map);


        // --------------------------------------------------
        // SET UP FILTER UI EVENTS
        // --------------------------------------------------
        // noUiSlider initialization (after script load)
        function initTimeSlider() {
            var slider = document.getElementById('timeSlider');
            if (!slider || slider.noUiSlider) {
                return;  // already initialized or missing
            }
            if (!window.noUiSlider) {
                return;
            }

            window.noUiSlider.create(slider, {
                start: [filterState.minSec, filterState.maxSec],
                connect: true,
                range: {
                    'min': globalMinSec,
                    'max': globalMaxSec
                },
                step: 1,
                tooltips: false,   // no labels on handles, only label below
                format: {
                    to: function(v) { return Math.round(v); },
                    from: function(v) { return Number(v); }
                }
            });

            slider.noUiSlider.on('update', function(values) {
                var vMin = Number(values[0]);
                var vMax = Number(values[1]);

                filterState.minSec = vMin;
                filterState.maxSec = vMax;

                var lbl = document.getElementById('timeRangeLabel');
                if (lbl) {
                    lbl.textContent = vMin + " – " + vMax;
                }

                if (window._lastStartId && window._lastStartMarker) {
                    highlightStart(window._lastStartId, window._lastStartMarker);
                }
            });
        }

        // try to init immediately if noUiSlider is there
        if (window.noUiSlider) {
            initTimeSlider();
        } else if (window._noUiSliderScriptRef) {
            window._noUiSliderScriptRef.addEventListener('load', initTimeSlider);
        } else {
            // fallback: small timeout if something odd happens
            setTimeout(initTimeSlider, 700);
        }

        // Bike / rider / month filter events
        var chkClassic  = document.getElementById("chkClassic");
        var chkElectric = document.getElementById("chkElectric");
        var chkMember   = document.getElementById("chkMember");
        var chkCasual   = document.getElementById("chkCasual");
        var monthSelect = document.getElementById("monthSelect");
        var resetBtnEl  = document.getElementById("resetMapBtn");

        if (chkClassic) {
            chkClassic.onchange = function() {
                filterState.bikeTypes.classic_bike = this.checked;

                if (window._lastStartId && window._lastStartMarker) {
                    // Detail mode: recompute endpoints for selected start
                    highlightStart(window._lastStartId, window._lastStartMarker);
                } else {
                    // Overview mode: change which start stations appear
                    filterStartsByFilters();
                }
            };
        }
        if (chkElectric) {
            chkElectric.onchange = function() {
                filterState.bikeTypes.electric_bike = this.checked;

                if (window._lastStartId && window._lastStartMarker) {
                    highlightStart(window._lastStartId, window._lastStartMarker);
                } else {
                    filterStartsByFilters();
                }
            };
        }
        if (chkMember) {
            chkMember.onchange = function() {
                filterState.memberTypes.member = this.checked;

                if (window._lastStartId && window._lastStartMarker) {
                    highlightStart(window._lastStartId, window._lastStartMarker);
                } else {
                    filterStartsByFilters();
                }
            };
        }
        if (chkCasual) {
            chkCasual.onchange = function() {
                filterState.memberTypes.casual = this.checked;

                if (window._lastStartId && window._lastStartMarker) {
                    highlightStart(window._lastStartId, window._lastStartMarker);
                } else {
                    filterStartsByFilters();
                }
            };
        }
        if (monthSelect) {
            monthSelect.onchange = function() {
                filterState.month = this.value;

                // NEW LOGIC:
                // - If a station is selected, just re-highlight that station
                //   so its endpoints get month-filtered (stay in detail view).
                // - If no station selected, we are in overview → filter which
                //   starts are visible in the cluster.
                if (window._lastStartId && window._lastStartMarker) {
                    highlightStart(window._lastStartId, window._lastStartMarker);
                } else {
                    resetMap();
                    filterStartsByMonth();
                }
            };
        }

        // Reset button inside filter box: reset everything
        if (resetBtnEl) {
            resetBtnEl.onclick = function() {
                // 1) reset spatial selection
                resetMap();

                // 2) reset filter state
                filterState.minSec = globalMinSec;
                filterState.maxSec = globalMaxSec;
                filterState.bikeTypes.classic_bike = true;
                filterState.bikeTypes.electric_bike = true;
                filterState.memberTypes.member = true;
                filterState.memberTypes.casual = true;
                filterState.month = "ALL";

                // 3) reset UI controls
                var slider = document.getElementById('timeSlider');
                if (slider && slider.noUiSlider) {
                    slider.noUiSlider.set([globalMinSec, globalMaxSec]);
                }
                if (chkClassic)  chkClassic.checked  = true;
                if (chkElectric) chkElectric.checked = true;
                if (chkMember)   chkMember.checked   = true;
                if (chkCasual)   chkCasual.checked   = true;
                if (monthSelect) monthSelect.value   = "ALL";

                var lbl = document.getElementById('timeRangeLabel');
                if (lbl) {
                    lbl.textContent = globalMinSec + " – " + globalMaxSec;
                }

                // 4) show all start markers again (overview)
                filterStartsByFilters();
            };
        }

        // --------------------------------------------------
        // REGISTER START MARKERS
        // --------------------------------------------------
        {% for item in this.bindings %}
            window._allStartMarkers.push({{ item.marker_name }});

            // tag each marker with its startId for month filtering
            {{ item.marker_name }}._startId = "{{ item.start_id }}";

            {{ item.marker_name }}.on('click', function(e) {
                window._lastStartId     = "{{ item.start_id }}";
                window._lastStartMarker = this;
                highlightStart("{{ item.start_id }}", this);
            });
        {% endfor %}

        // Initial overview: show all stations (month = ALL)
        filterStartsByFilters();

        {% endmacro %}
    """)


    def __init__(
        self,
        rides_by_start,
        bindings,
        cluster_name,
        bike_icon_url,
        flag_icon_url,
        selected_popups,
        min_sec,
        max_sec,
        months
    ):
        super().__init__()
        self.rides_by_start = rides_by_start
        self.bindings = bindings
        self.cluster_name = cluster_name
        self.bike_icon_url = bike_icon_url
        self.flag_icon_url = flag_icon_url
        self.selected_popups = selected_popups
        self.min_sec = min_sec
        self.max_sec = max_sec
        self.months = months


In [11]:
def create_ride_map(
    df,
    html_path="rides_map.html",
    sample_n=10000,
    min_sec=None,
    max_sec=None,
    nyc_only=True,
    return_map=False,
    use_sample=False
):
    # optional filtering / sampling (use if you want)
    tmp = df.copy()

    if min_sec is not None:
        tmp = tmp[tmp["ride_time_seconds"] >= min_sec]
    if max_sec is not None:
        tmp = tmp[tmp["ride_time_seconds"] <= max_sec]

    if use_sample and sample_n is not None and len(tmp) > sample_n:
        tmp = tmp.sample(sample_n, random_state=42)

    # after tmp is defined / filtered
    if "month" in tmp.columns:
        unique_months = sorted(tmp["month"].dropna().unique().tolist())
    else:
        unique_months = []   # fallback if no month column

    # base map
    m = folium.Map(
        location=[40.75, -73.98],
        zoom_start=12,
        tiles="CartoDB Positron",
    )

    cluster = MarkerCluster(name="Start stations").add_to(m)

    # Group rides by start location
    stations = {}
    for _, row in tmp.iterrows():
        key = (round(row["start_lat"], 5), round(row["start_lng"], 5))

        if key not in stations:
            stations[key] = {
                "start_lat": row["start_lat"],
                "start_lng": row["start_lng"],
                "start_station_name": row.get("start_station_name", ""),
                "rides": []
            }

        stations[key]["rides"].append({
            "lat": row["end_lat"],
            "lng": row["end_lng"],
            "ride_id": row.get("ride_id", None),
            "duration_sec": row["ride_time_seconds"],
            "duration_min": row["ride_time_seconds"] / 60.0,
            "member_casual": row.get("member_casual", ""),
            "rideable_type": row.get("rideable_type", ""),
            "end_station_name": row.get("end_station_name", ""),
            "month": row.get("month", None)
        })

    all_secs = [r["duration_sec"] for s in stations.values() for r in s["rides"]]
    min_sec_all = min(all_secs) if all_secs else 0
    max_sec_all = max(all_secs) if all_secs else 7200   # 2 hours

    # Create markers (one per start) and build JS data
    bindings = []         # mapping of folium marker -> start_id
    rides_by_start_dict = {}
    selected_popup_dict = {}

    for idx, ((lat_key, lng_key), info) in enumerate(stations.items()):
        start_lat = info["start_lat"]
        start_lng = info["start_lng"]
        start_name = info["start_station_name"]
        start_id = f"start_{idx}"  # simple string ID used in JS

        # store all endpoints for this start in JS dict
        rides_by_start_dict[start_id] = [
            {
                "lat": r["lat"],
                "lng": r["lng"],
                "name": r.get("end_station_name", ""),
                "sec": r["duration_sec"],
                "bike_type": r.get("rideable_type", ""),
                "member": r.get("member_casual", ""),
                "month": r.get("month", None)
            }
            for r in info["rides"]
        ]

        unique_months = sorted(tmp["month"].dropna().unique().tolist())

        # popup (you can customize; here: show number of rides from this start)
        popup_html = f"""
        <b>Start station:</b> {start_name}<br>
        <b>Start coord:</b> ({start_lat:.5f}, {start_lng:.5f})<br>
        <b>Outgoing rides:</b> {len(info['rides'])} rides<br>
        """

        selected_popup_dict[start_id] = popup_html

        start_marker = folium.Marker(
            location=(start_lat, start_lng),
            popup=folium.Popup(popup_html, max_width=300),
            tooltip=start_name or "Start station",
            icon=folium.Icon(color="black", icon="bicycle", prefix="fa"),
        ).add_to(cluster)

        # record mapping so JS can bind click handler
        bindings.append({
            "marker_name": start_marker.get_name(),  # JS variable name for this marker
            "start_id": start_id
        })

    # Attach custom JS to map
    rides_by_start_json = json.dumps(rides_by_start_dict)
    js = RideHighlightJS(
        rides_by_start=rides_by_start_json,
        bindings=bindings,
        cluster_name=cluster.get_name(),
        bike_icon_url="../images/bike_icon.png",
        flag_icon_url="../images/flag_icon.png",
        selected_popups=json.dumps(selected_popup_dict),
        min_sec=min_sec_all,
        max_sec=max_sec_all,
        months=json.dumps(unique_months),
    )
    m.add_child(js)

    # Save / return
    m.save(html_path)
    if return_map:
        return m

In [12]:
create_ride_map(
    df_reduced,
    html_path=f"{OUTPUT_PATH}/2023_NYC_Interactive_Rides_Map.html",
    min_sec=0,
    max_sec=2*3600,
    nyc_only=True,
)